# RodPrep — exploration

Notebook d'exploration de l'étape 1 : extraction du récap Excel ROD et construction de la table hôtel.

Objectif : visualiser les entrées, les étapes intermédiaires et remplir `../Output/`.

In [49]:
from pathlib import Path
import sys

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

ROOT = Path.cwd().resolve()
while ROOT.name != "RodPrep" and ROOT.parent != ROOT:
    ROOT = ROOT.parent
PREPARE = ROOT.parent
PROJECT = PREPARE.parent

sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(ROOT / "Src"))

INPUT_DIR = ROOT / "Input"
OUTPUT_DIR = ROOT / "Output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 1. Entrée — récap Excel et registre identité

In [50]:
from rod_ia.config.settings import get_settings
from rod_ia.domain.repositories.identity_registry import HotelIdentityRegistry
from rod_prep.prep import RodPrep

settings = get_settings(PROJECT)
prep = RodPrep(INPUT_DIR, OUTPUT_DIR, settings.identity_registry_path)

recap_path = prep.seed_input_from_sources()
print("Fichier récap :", recap_path)

registry = HotelIdentityRegistry(settings.identity_registry_path)
registry_df = pd.DataFrame([r.to_dict() for r in registry.all_records()])
print(f"Registre identité : {len(registry_df)} hôtels")
display(registry_df.head(3))
registry_df[["hotel_id", "name_ventes", "brand", "city", "nb_chambres"]].head(10)

Fichier récap : /media/laghmari/ssd-data/dev/hotels/prepare/RodPrep/Input/recapitulatif_rod.xlsx
Registre identité : 8 hôtels


,hotel_id,brand,city,name_display,name_ventes,name_rod,aliases,lat_canonical,lon_canonical,geo_source,lat_rod,lon_rod,lat_nominatim,lon_nominatim,has_sales,has_rod,nb_chambres
0,ibis-budget-nice,IBIS BUDGET,Nice,Ibis budget Nice Californie,Ibis budget Nice,Nice Californie,"[Ibis Budget Nice, IBIS BUDGET Nice]",43.710000,7.260000,nominatim,None,None,43.689258,7.240379,True,True,129.0
1,ibis-budget-strasbourg,IBIS BUDGET,Strasbourg,Ibis budget Strasbourg Centre République,Ibis budget Strasbourg Centre République,Strasbourg République,"[Ibis Budget Strasbourg, Ibis budget Strasbourg]",NaN,NaN,None,None,None,NaN,NaN,True,True,97.0
2,ibis-styles-roissy-cdg,IBIS STYLES,Roissy,Ibis Styles Roissy CDG,None,Roissy CDG,[],49.007078,2.520403,nominatim,None,None,49.007078,2.520403,False,True,309.0


,hotel_id,name_ventes,brand,city,nb_chambres
0,ibis-budget-nice,Ibis budget Nice,IBIS BUDGET,Nice,129.0
1,ibis-budget-strasbourg,Ibis budget Strasbourg Centre République,IBIS BUDGET,Strasbourg,97.0
2,ibis-styles-roissy-cdg,None,IBIS STYLES,Roissy,309.0
3,novotel-megeve,Novotel Megève Mont-Blanc,NOVOTEL,Megève,572.0
4,novotel-paris-tour-eiffel,Novotel Paris Tour Eiffel,NOVOTEL,Paris,764.0
5,mercure-montmartre,Mercure Paris Montmartre Sacré-Cœur,MERCURE,Paris,305.0
6,mercure-boulogne,None,MERCURE,Boulogne-Billancourt,191.0
7,novotel-porte-italie,Novotel Porte d'Italie,NOVOTEL,Paris,NaN


## 2. Extraction longue — une ligne par variable × hôtel

In [51]:
from rod_ia.domain.services.rod_recap_extractor import RodRecapExtractor

extractor = RodRecapExtractor(
    recap_path=recap_path,
    identity_registry=registry,
    output_path=OUTPUT_DIR / "rod_recap",
)

long_df = extractor.extract_long()
print(f"Format long : {long_df.shape[0]} lignes × {long_df.shape[1]} colonnes")
long_df.head(12)

Format long : 938 lignes × 9 colonnes


,hotel_id,recap_column,row,etape,sous_etape,data_label,field_key,field_type_hint,raw_value
0,ibis-budget-nice,NICE,4,0 - PAGE DE CONNEXION,ID,CODE H,0_page_de_connexion_id_code_h,categorical,H2075
1,ibis-budget-strasbourg,STRASBOURG,4,0 - PAGE DE CONNEXION,ID,CODE H,0_page_de_connexion_id_code_h,categorical,HB6A3
2,ibis-styles-roissy-cdg,PARIS CDG,4,0 - PAGE DE CONNEXION,ID,CODE H,0_page_de_connexion_id_code_h,categorical,H0815
3,novotel-megeve,MEGEVE,4,0 - PAGE DE CONNEXION,ID,CODE H,0_page_de_connexion_id_code_h,categorical,HB5I0
4,novotel-paris-tour-eiffel,TOUR EIFFEL,4,0 - PAGE DE CONNEXION,ID,CODE H,0_page_de_connexion_id_code_h,categorical,H3546
5,mercure-montmartre,MONTMARTRE,4,0 - PAGE DE CONNEXION,ID,CODE H,0_page_de_connexion_id_code_h,categorical,H0373
6,mercure-boulogne,BOULOGNE,4,0 - PAGE DE CONNEXION,ID,CODE H,0_page_de_connexion_id_code_h,categorical,H6188
7,ibis-budget-nice,NICE,5,0 - PAGE DE CONNEXION,ID,NOM DE L'HOTEL,0_page_de_connexion_id_nom_de_l_hotel,categorical,IBIS BUDGET NICE CALIFORNIE
8,ibis-budget-strasbourg,STRASBOURG,5,0 - PAGE DE CONNEXION,ID,NOM DE L'HOTEL,0_page_de_connexion_id_nom_de_l_hotel,categorical,IBIS BUDGET STRASBOURG REPUBLIQUE
9,ibis-styles-roissy-cdg,PARIS CDG,5,0 - PAGE DE CONNEXION,ID,NOM DE L'HOTEL,0_page_de_connexion_id_nom_de_l_hotel,categorical,IBIS STYLES ROISSY CDG


## 3. Format wide — features `d_recap_*` par hôtel

In [52]:
wide_df = extractor.extract_wide()
print(f"Format wide : {wide_df.shape[0]} hôtels × {wide_df.shape[1]} colonnes")
wide_df.head()

Format wide : 7 hôtels × 90 colonnes


,hotel_id,d_recap_0_page_de_connexion_localisation_geo_adresse_postale_1,d_recap_0_page_de_connexion_localisation_geo_adresse_postale_2,d_recap_0_page_de_connexion_localisation_geo_code_postal,d_recap_0_page_de_connexion_localisation_geo_latitude,d_recap_0_page_de_connexion_localisation_geo_longitude,d_recap_0_page_de_connexion_localisation_geo_ville,d_recap_2_services_equipements_dispo_dans_le_lobby_assises,d_recap_2_services_equipements_dispo_dans_le_lobby_bouilloire,d_recap_2_services_equipements_dispo_dans_le_lobby_fontaine_a_eau,d_recap_2_services_equipements_dispo_dans_le_lobby_machine_a_cafe,d_recap_2_services_equipements_dispo_dans_le_lobby_micro_ondes,d_recap_2_services_equipements_dispo_dans_le_lobby_vitrine_refrigeree,d_recap_2_services_equipements_f_b_bar,d_recap_2_services_equipements_f_b_horaires_d_ouverture,d_recap_2_services_equipements_f_b_horaires_d_ouverture_r43,d_recap_2_services_equipements_f_b_jours_d_ouverture,d_recap_2_services_equipements_f_b_jours_d_ouverture_r44,d_recap_2_services_equipements_f_b_minibar,d_recap_2_services_equipements_f_b_mois_d_ouverture,d_recap_2_services_equipements_f_b_mois_d_ouverture_r45,d_recap_2_services_equipements_f_b_restaurant,d_recap_2_services_equipements_f_b_room_service,d_recap_2_services_equipements_non_f_b_piscine,d_recap_2_services_equipements_non_f_b_salle_de_sport,d_recap_2_services_equipements_non_f_b_salles_de_reunion,d_recap_2_services_equipements_non_f_b_spa,d_recap_3_profil_de_vos_clients_affaires_affaires_pct,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_boissons_alcoolisees,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_boissons_non_alcoolisees,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_epicerie_fine,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_produits_sales_frais,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_produits_sales_secs,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_produits_sucres_frais,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_produits_sucres_secs,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_accessoires,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_articles_pour_enfants,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_hygiene,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_pret_a_porter,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_produits_cosmetiques,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_produits_sos,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_souvenirs,d_recap_3_profil_de_vos_clients_loisirs_loisirs_pct,d_recap_3_profil_de_vos_clients_loisirs_top_1_amis,d_recap_3_profil_de_vos_clients_loisirs_top_1_couples,d_recap_3_profil_de_vos_clients_loisirs_top_1_familles,d_recap_3_profil_de_vos_clients_national_vs_inter_international_pct,d_recap_3_profil_de_vos_clients_national_vs_inter_national_pct,d_recap_corner_autre_emplacement_dispo_metres_carres_estimation,d_recap_corner_autre_emplacement_dispo_metres_lineaires_estimation,d_recap_corner_autre_emplacement_dispo_xpct_de_mes_clients_passent_devant,d_recap_corner_corner_de_vente_actuel_metres_carres_estimation,d_recap_corner_corner_de_vente_actuel_metres_lineaires_estimation,d_recap_corner_corner_de_vente_actuel_votre_hotel_dispose_t_il_deja_d_un,d_recap_corner_equipements_disponibles_alimentation_electrique_r124,d_recap_corner_equipements_disponibles_internet_filaire_prise_rj45_r123,d_recap_corner_equipements_disponibles_videosurveillance_r125,d_recap_corner_equipements_disponibles_wifi_r122,d_recap_corner_votre_corner_actuel_offre_f_b_caisse_code_barres,d_recap_corner_votre_corner_actuel_offre_f_b_distributeur_auto,d_recap_corner_votre_corner_actuel_offre_f_b_frigo_connecte,d_recap_corner_votre_corner_actuel_offre_f_b_hotel_staff,d_recap_corner_votre_corner_actuel_offre_f_b_liste_des_produits_f_b,d_recap_corner_votre_corner_actuel_offre_f_b_reception,d_recap_corner_votre_corner_ac

## 4. Table de liaison `hotel_lookup`

In [53]:
hotel_lookup = prep.run()  # persiste aussi rod_features + hotel_lookup
print(f"hotel_lookup : {hotel_lookup.shape}")
hotel_lookup.head()

hotel_lookup : (8, 95)


,hotel_code,hotel_name,nom_hotel,hotel_brand,hotel_city,nb_chambres,d_recap_0_page_de_connexion_localisation_geo_adresse_postale_1,d_recap_0_page_de_connexion_localisation_geo_adresse_postale_2,d_recap_0_page_de_connexion_localisation_geo_code_postal,d_recap_0_page_de_connexion_localisation_geo_ville,d_recap_2_services_equipements_dispo_dans_le_lobby_assises,d_recap_2_services_equipements_dispo_dans_le_lobby_bouilloire,d_recap_2_services_equipements_dispo_dans_le_lobby_fontaine_a_eau,d_recap_2_services_equipements_dispo_dans_le_lobby_machine_a_cafe,d_recap_2_services_equipements_dispo_dans_le_lobby_micro_ondes,d_recap_2_services_equipements_dispo_dans_le_lobby_vitrine_refrigeree,d_recap_2_services_equipements_f_b_bar,d_recap_2_services_equipements_f_b_horaires_d_ouverture,d_recap_2_services_equipements_f_b_horaires_d_ouverture_r43,d_recap_2_services_equipements_f_b_jours_d_ouverture,d_recap_2_services_equipements_f_b_jours_d_ouverture_r44,d_recap_2_services_equipements_f_b_minibar,d_recap_2_services_equipements_f_b_mois_d_ouverture,d_recap_2_services_equipements_f_b_mois_d_ouverture_r45,d_recap_2_services_equipements_f_b_restaurant,d_recap_2_services_equipements_f_b_room_service,d_recap_2_services_equipements_non_f_b_piscine,d_recap_2_services_equipements_non_f_b_salle_de_sport,d_recap_2_services_equipements_non_f_b_salles_de_reunion,d_recap_2_services_equipements_non_f_b_spa,d_recap_3_profil_de_vos_clients_affaires_affaires_pct,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_boissons_alcoolisees,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_boissons_non_alcoolisees,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_epicerie_fine,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_produits_sales_frais,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_produits_sales_secs,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_produits_sucres_frais,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_produits_sucres_secs,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_accessoires,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_articles_pour_enfants,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_hygiene,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_pret_a_porter,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_produits_cosmetiques,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_produits_sos,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_souvenirs,d_recap_3_profil_de_vos_clients_loisirs_loisirs_pct,d_recap_3_profil_de_vos_clients_loisirs_top_1_amis,d_recap_3_profil_de_vos_clients_loisirs_top_1_couples,d_recap_3_profil_de_vos_clients_loisirs_top_1_familles,d_recap_3_profil_de_vos_clients_national_vs_inter_international_pct,d_recap_3_profil_de_vos_clients_national_vs_inter_national_pct,d_recap_corner_autre_emplacement_dispo_metres_carres_estimation,d_recap_corner_autre_emplacement_dispo_metres_lineaires_estimation,d_recap_corner_autre_emplacement_dispo_xpct_de_mes_clients_passent_devant,d_recap_corner_corner_de_vente_actuel_metres_carres_estimation,d_recap_corner_corner_de_vente_actuel_metres_lineaires_estimation,d_recap_corner_corner_de_vente_actuel_votre_hotel_dispose_t_il_deja_d_un,d_recap_corner_equipements_disponibles_alimentation_electrique_r124,d_recap_corner_equipements_disponibles_internet_filaire_prise_rj45_r123,d_recap_corner_equipements_disponibles_videosurveillance_r125,d_recap_corner_equipements_disponibles_wifi_r122,d_recap_corner_votre_corner_actuel_offre_f_b_caisse_code_barres,d_recap_corner_votre_corner_actuel_offre_f_b_distributeur_auto,d_recap_corner_votre_corner_actuel_offre_f_b_frigo_connecte,d_recap_corner_votre_corner_actuel_offre_f_b_hotel_staff,d_recap_corner_votre_corner_actuel_offre_f_b_liste_des_produits_f_b,d_recap_corner_votre_corner_actuel_offre_f_b_reception,d_recap_corner_votre_corner_actuel_offre_f_b_snacking_comptoir,d_recap_corner_vot

## 5. Aperçu colonnes récap retenues

In [54]:
recap_cols = [c for c in hotel_lookup.columns if str(c).startswith("d_recap_")]
print(f"{len(recap_cols)} colonnes d_recap_")
if recap_cols:
    hotel_lookup[["hotel_code", "nom_hotel"] + recap_cols[:8]].head()

86 colonnes d_recap_


## 6. Entrée MeteoPrep / ProximityPrep

In [55]:
meteo_input = prep.to_meteo_input()
meteo_input

,hotel_code,hotel_name,hotel_brand,hotel_city,hotel_lat,hotel_lon
0,H2075,Ibis budget Nice Californie,IBIS BUDGET,Nice,43.689186,7.240512
1,HB6A3,Ibis budget Strasbourg Centre République,IBIS BUDGET,Strasbourg,48.591522,7.754599
2,H0815,Ibis Styles Roissy CDG,IBIS STYLES,Roissy,49.006733,2.519843
3,H6188,Mercure Paris Boulogne,MERCURE,Boulogne-Billancourt,48.833827,2.256274
4,H0373,Mercure Paris Montmartre Sacré-Cœur,MERCURE,Paris,48.885048,2.329923
5,HB5I0,Novotel Megève Mont-Blanc,NOVOTEL,Megève,45.859165,6.619055
6,H3546,Novotel Paris Centre Tour Eiffel,NOVOTEL,Paris,48.849778,2.282836


## 7. Fichiers produits dans Output/

In [56]:
for path in sorted(OUTPUT_DIR.glob("*")):
    print(path.name)

hotel_lookup.csv
hotel_lookup.parquet
rod_features.csv
rod_features.parquet
rod_recap.long.csv
rod_recap.schema.json
rod_recap.wide.csv


In [57]:
# df = hotel_lookup

In [58]:
unwanted_prefixes= [
    "d_recap_0_page_de_connexion_localisation_geo_",
    "d_recap_2_services_equipements_",
    "d_recap_3_profil_de_vos_clients_affaires_" ,
    "d_recap_3_profil_de_vos_clients_affaires_",
    "d_recap_3_profil_de_vos_clients_",
    "d_recap_corner_",
    "d_recap_de_controle_parametres_",
    "d_recap_generales_donnees_admin_",
    "d_recap_generales_donnees_chiffrees_",
    ""
]


hotel_lookup.columns = hotel_lookup.columns.str.replace(
    "|".join(unwanted_prefixes), 
    "", 
    regex=True
).str.strip().str.replace(r'_+', '_', regex=True).str.strip('_')

In [59]:
hotel_lookup

,hotel_code,hotel_name,nom_hotel,hotel_brand,hotel_city,nb_chambres,adresse_postale_1,adresse_postale_2,code_postal,ville,dispo_dans_le_lobby_assises,dispo_dans_le_lobby_bouilloire,dispo_dans_le_lobby_fontaine_a_eau,dispo_dans_le_lobby_machine_a_cafe,dispo_dans_le_lobby_micro_ondes,dispo_dans_le_lobby_vitrine_refrigeree,f_b_bar,f_b_horaires_d_ouverture,f_b_horaires_d_ouverture_r43,f_b_jours_d_ouverture,f_b_jours_d_ouverture_r44,f_b_minibar,f_b_mois_d_ouverture,f_b_mois_d_ouverture_r45,f_b_restaurant,f_b_room_service,non_f_b_piscine,non_f_b_salle_de_sport,non_f_b_salles_de_reunion,non_f_b_spa,affaires_pct,besoins_de_vos_clients_f_b_boissons_alcoolisees,besoins_de_vos_clients_f_b_boissons_non_alcoolisees,besoins_de_vos_clients_f_b_epicerie_fine,besoins_de_vos_clients_f_b_produits_sales_frais,besoins_de_vos_clients_f_b_produits_sales_secs,besoins_de_vos_clients_f_b_produits_sucres_frais,besoins_de_vos_clients_f_b_produits_sucres_secs,besoins_de_vos_clients_non_f_b_accessoires,besoins_de_vos_clients_non_f_b_articles_pour_enfants,besoins_de_vos_clients_non_f_b_hygiene,besoins_de_vos_clients_non_f_b_pret_a_porter,besoins_de_vos_clients_non_f_b_produits_cosmetiques,besoins_de_vos_clients_non_f_b_produits_sos,besoins_de_vos_clients_non_f_b_souvenirs,loisirs_loisirs_pct,loisirs_top_1_amis,loisirs_top_1_couples,loisirs_top_1_familles,national_vs_inter_international_pct,national_vs_inter_national_pct,autre_emplacement_dispo_metres_carres_estimation,autre_emplacement_dispo_metres_lineaires_estimation,autre_emplacement_dispo_xpct_de_mes_clients_passent_devant,corner_de_vente_actuel_metres_carres_estimation,corner_de_vente_actuel_metres_lineaires_estimation,corner_de_vente_actuel_votre_hotel_dispose_t_il_deja_d_un,equipements_disponibles_alimentation_electrique_r124,equipements_disponibles_internet_filaire_prise_rj45_r123,equipements_disponibles_videosurveillance_r125,equipements_disponibles_wifi_r122,votre_corner_actuel_offre_f_b_caisse_code_barres,votre_corner_actuel_offre_f_b_distributeur_auto,votre_corner_actuel_offre_f_b_frigo_connecte,votre_corner_actuel_offre_f_b_hotel_staff,votre_corner_actuel_offre_f_b_liste_des_produits_f_b,votre_corner_actuel_offre_f_b_reception,votre_corner_actuel_offre_f_b_snacking_comptoir,votre_corner_actuel_offre_non_f_b_armoire_connectee,votre_corner_actuel_offre_non_f_b_caisse_code_barres,votre_corner_actuel_offre_non_f_b_distributeur_auto,votre_corner_actuel_offre_non_f_b_hotel_staff,votre_corner_actuel_offre_non_f_b_liste_des_produits_non_f,votre_corner_actuel_offre_non_f_b_reception,metres_lineaires_dedies_a_v,moyen_de_guests_par_chambre,nb_de_chambres,to_annuel_moyen,contrat_signe_annee,contrat_type,derniere_reno_hotel,derniere_reno_lobby,dom_dof,marque,nb_de_chambres,pms,proprietaire,to_annuel,to_le_plus_bas_mois,to_le_plus_bas_taux,to_le_plus_haut_mois,to_le_plus_haut_taux,hotel_lat,hotel_lon,hotel_geo_source
0,H2075,Ibis budget Nice Californie,Ibis budget Nice,IBIS BUDGET,Nice,129.0,58-60 AVENUE DE LA CALIFORNIE,NaN,06200,NICE,1.0,NaN,1.0,1.0,1.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,4.0,0.0,NaN,NaN,1.0,NaN,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,1.0,1.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,3.0,1.0,NaN,NaN,NaN,NaN,0.0,1.0,0.0,NaN,1.0,0.0,0.0,0.0,0.0,1.0,NaN,0.0,0.0,6.0,NaN,129.0,NaN,NaN,FRANCHISE,NaN,NaN,Marion BOROT,IBIS BUDGET,129.0,FOLS,FAMILLE FARINES,NaN,NaN,NaN,NaN,NaN,43.689186,7.240512,recap
1,HB6A3,Ibis budget Strasbourg Centre République,Ibis budget Strasbourg Centre République,IBIS BUDGET,Strasbourg,97.0,23A RUE OBERLIN,NaN,67000,STRASBOURG,1.0,0.0,1.0,1.0,0.0,0.0,0.0,16h - 00h,NaN,MARDI AU SAMEDI,NaN,1.0,TOUS,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.4,NaN,1.0,NaN,1.0,1.0,1.0,1.0,NaN,NaN,1.0,NaN,NaN,1.0,NaN,0.6,1.0,NaN,NaN,0.4,0.6,3.0,3.0,100.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,2.0,1.5,97.0,0.70,2020.0,FRANCHISE,2025.0,2025.0,Christophe RUQUEBOEUCHE,IBIS BUDGET,97.0,OPERA,MILLION - CHOREIN,0.70,JANVIER,0.60,DECEMBRE,0.80,48.591522,7.754599,r

In [60]:
map_cols = {'hotel_code': 'hotel_code',
 'hotel_name': 'hotel_name',
#  'nom_hotel': 'nom_hotel',
 'hotel_brand': 'hotel_brand',
 'hotel_city': 'hotel_city',
 'adresse_postale_1': 'hotel_adresse_postale_1',
 'adresse_postale_2': 'hotel_adresse_postale_2',
 'code_postal': 'hotel_code_postal',
#  'ville': 'ville',

 'hotel_lat': 'hotel_lat',
 'hotel_lon': 'hotel_lon',



#   'marque': 'hotel_marque',
 'contrat_signe_annee': 'hotel_contrat_signe_annee',
 'contrat_type': 'hotel_contrat_type',
 'derniere_reno_hotel': 'hotel_derniere_reno',
 'derniere_reno_lobby': 'hotel_lobby_derniere_reno',
#  'dom_dof': 'dom_dof',
#  'pms': 'hotel_pms',
#  'proprietaire': 'hotel_proprietaire',
 'nb_chambres': 'hotel_nb_chambres',
 'to_annuel': 'hotel_to_annuel',
 # 'nb_de_chambres': 'hotel_nb_de_chambres',
#  'moyen_de_guests_par_chambre': 'hotel_moyen_de_guests_par_chambre',
#  'to_annuel_moyen': 'hotel_to_annuel_moyen',

#  'to_le_plus_bas_mois': 'hotel_to_le_plus_bas_mois',
 'to_le_plus_bas_taux': 'hotel_to_le_plus_bas_taux',
#  'to_le_plus_haut_mois': 'hotel_to_le_plus_haut_mois',
 'to_le_plus_haut_taux': 'hotel_to_le_plus_haut_taux',





 'dispo_dans_le_lobby_assises': 'hotel_dispo_dans_lobby_assises',
 'dispo_dans_le_lobby_bouilloire': 'hotel_dispo_dans_lobby_bouilloire',
 'dispo_dans_le_lobby_fontaine_a_eau': 'hotel_dispo_dans_lobby_fontaine_a_eau',
 'dispo_dans_le_lobby_machine_a_cafe': 'hotel_dispo_dans_lobby_machine_a_cafe',
 'dispo_dans_le_lobby_micro_ondes': 'hotel_dispo_dans_lobby_micro_ondes',
 'dispo_dans_le_lobby_vitrine_refrigeree': 'hotel_dispo_dans_lobby_vitrine_refrigeree',
 
 'f_b_bar': 'hotel_f_b_bar',
#  'f_b_horaires_d_ouverture': 'f_b_horaires_d_ouverture',
#  'f_b_horaires_d_ouverture_r43': 'f_b_horaires_d_ouverture_r43',
#  'f_b_jours_d_ouverture': 'f_b_jours_d_ouverture',
#  'f_b_jours_d_ouverture_r44': 'f_b_jours_d_ouverture_r44',
 'f_b_minibar': 'hotel_f_b_minibar',
#  'f_b_mois_d_ouverture': 'f_b_mois_d_ouverture',
#  'f_b_mois_d_ouverture_r45': 'f_b_mois_d_ouverture_r45',
 'f_b_restaurant': 'hotel_f_b_restaurant',
 'f_b_room_service': 'hotel_f_b_room_service',

 'non_f_b_piscine': 'hotel_non_f_b_piscine',
 'non_f_b_salle_de_sport': 'hotel_non_f_b_salle_de_sport',
 'non_f_b_salles_de_reunion': 'hotel_non_f_b_salles_de_reunion',
 'non_f_b_spa': 'hotel_non_f_b_spa',



#  'besoins_de_vos_clients_f_b_boissons_alcoolisees': 'besoins_f_b_boissons_alcoolisees',
#  'besoins_de_vos_clients_f_b_boissons_non_alcoolisees': 'besoins_f_b_boissons_non_alcoolisees',
#  'besoins_de_vos_clients_f_b_epicerie_fine': 'besoins_f_b_epicerie_fine',
#  'besoins_de_vos_clients_f_b_produits_sales_frais': 'besoins_f_b_produits_sales_frais',
#  'besoins_de_vos_clients_f_b_produits_sales_secs': 'besoins_f_b_produits_sales_secs',
#  'besoins_de_vos_clients_f_b_produits_sucres_frais': 'besoins_f_b_produits_sucres_frais',
#  'besoins_de_vos_clients_f_b_produits_sucres_secs': 'besoins_f_b_produits_sucres_secs',

#  'besoins_de_vos_clients_non_f_b_accessoires': 'besoins_non_f_b_accessoires',
#  'besoins_de_vos_clients_non_f_b_articles_pour_enfants': 'besoins_non_f_b_articles_pour_enfants',
#  'besoins_de_vos_clients_non_f_b_hygiene': 'besoins_non_f_b_hygiene',
#  'besoins_de_vos_clients_non_f_b_pret_a_porter': 'besoins_non_f_b_pret_a_porter',
#  'besoins_de_vos_clients_non_f_b_produits_cosmetiques': 'besoins_non_f_b_produits_cosmetiques',
#  'besoins_de_vos_clients_non_f_b_produits_sos': 'besoins_non_f_b_produits_sos',
#  'besoins_de_vos_clients_non_f_b_souvenirs': 'besoins_non_f_b_souvenirs',

 'affaires_pct': 'hotel_affaires_pct',
 'loisirs_loisirs_pct': 'hotel_loisirs_pct',
 'loisirs_top_1_amis': 'hotel_loisirs_top_1_amis',
 'loisirs_top_1_couples': 'hotel_loisirs_top_1_couples',
 'loisirs_top_1_familles': 'hotel_loisirs_top_1_familles',

 'national_vs_inter_international_pct': 'hotel_international_pct',
 'national_vs_inter_national_pct': 'hotel_national_pct',
 
#  'autre_emplacement_dispo_metres_carres_estimation': 'autre_emplacement_dispo_metres_carres_estimation',
#  'autre_emplacement_dispo_metres_lineaires_estimation': 'autre_emplacement_dispo_metres_lineaires_estimation',
#  'autre_emplacement_dispo_xpct_de_mes_clients_passent_devant': 'autre_emplacement_dispo_xpct_de_mes_clients_passent_devant',
  
 'corner_de_vente_actuel_votre_hotel_dispose_t_il_deja_d_un': 'hotel_corner_actuel_existe_deja',
 'corner_de_vente_actuel_metres_lineaires_estimation': 'hotel_corner_de_vente_actuel_metres_lineaires',

 'votre_corner_actuel_offre_f_b_caisse_code_barres': 'hotel_corner_actuel_offre_f_b_caisse_code_barres',
 'votre_corner_actuel_offre_f_b_distributeur_auto': 'hotel_corner_actuel_offre_f_b_distributeur_auto',
 'votre_corner_actuel_offre_f_b_frigo_connecte': 'hotel_corner_actuel_offre_f_b_frigo_connecte',
 'votre_corner_actuel_offre_f_b_reception': 'hotel_corner_actuel_offre_f_b_reception',
 'votre_corner_actuel_offre_f_b_snacking_comptoir': 'hotel_corner_actuel_offre_f_b_snacking_comptoir',
#  'votre_corner_actuel_offre_f_b_hotel_staff': 'hohtel_corner_actuel_offre_f_b_reappro_hotel_staff',

#  'corner_de_vente_actuel_metres_carres_estimation': 'corner_de_vente_actuel_metres_carres_estimation',
 
 

#  'equipements_disponibles_alimentation_electrique_r124': 'equipements_disponibles_alimentation_electrique_r124',
#  'equipements_disponibles_internet_filaire_prise_rj45_r123': 'equipements_disponibles_internet_filaire_prise_rj45_r123',
#  'equipements_disponibles_videosurveillance_r125': 'equipements_disponibles_videosurveillance_r125',
#  'equipements_disponibles_wifi_r122': 'equipements_disponibles_wifi_r122',



#  'votre_corner_actuel_offre_f_b_liste_des_produits_f_b': 'votre_corner_actuel_offre_f_b_liste_des_produits_f_b',
 

 'votre_corner_actuel_offre_non_f_b_armoire_connectee': 'hotel_corner_actuel_offre_non_f_b_armoire_connectee',
 'votre_corner_actuel_offre_non_f_b_caisse_code_barres': 'hotel_corner_actuel_offre_non_f_b_caisse_code_barres',
 'votre_corner_actuel_offre_non_f_b_distributeur_auto': 'hotel_corner_actuel_offre_non_f_b_distributeur_auto',
#  'votre_corner_actuel_offre_non_f_b_hotel_staff': 'hotel_corner_actuel_offre_non_f_b_hotel_staff',
#  'votre_corner_actuel_offre_non_f_b_liste_des_produits_non_f': 'hotel_corner_actuel_offre_non_f_b_liste_des_produits_non_f',
 'votre_corner_actuel_offre_non_f_b_reception': 'hotel_corner_actuel_offre_non_f_b_reception',


 

 'metres_lineaires_dedies_a_v': 'hotel_metres_lineaires_dedies_corner',


#  'hotel_geo_source': 'hotel_geo_source'
 }

In [61]:
hotel_features = (
    hotel_lookup.iloc[:-1][list(map_cols.keys())]
    .rename(columns=map_cols)
    .copy()
)
hotel_features

,hotel_code,hotel_name,hotel_brand,hotel_city,hotel_adresse_postale_1,hotel_adresse_postale_2,hotel_code_postal,hotel_lat,hotel_lon,hotel_contrat_signe_annee,hotel_contrat_type,hotel_derniere_reno,hotel_lobby_derniere_reno,hotel_nb_chambres,hotel_to_annuel,hotel_to_le_plus_bas_taux,hotel_to_le_plus_haut_taux,hotel_dispo_dans_lobby_assises,hotel_dispo_dans_lobby_bouilloire,hotel_dispo_dans_lobby_fontaine_a_eau,hotel_dispo_dans_lobby_machine_a_cafe,hotel_dispo_dans_lobby_micro_ondes,hotel_dispo_dans_lobby_vitrine_refrigeree,hotel_f_b_bar,hotel_f_b_minibar,hotel_f_b_restaurant,hotel_f_b_room_service,hotel_non_f_b_piscine,hotel_non_f_b_salle_de_sport,hotel_non_f_b_salles_de_reunion,hotel_non_f_b_spa,hotel_affaires_pct,hotel_loisirs_pct,hotel_loisirs_top_1_amis,hotel_loisirs_top_1_couples,hotel_loisirs_top_1_familles,hotel_international_pct,hotel_national_pct,hotel_corner_actuel_existe_deja,hotel_corner_de_vente_actuel_metres_lineaires,hotel_corner_actuel_offre_f_b_caisse_code_barres,hotel_corner_actuel_offre_f_b_distributeur_auto,hotel_corner_actuel_offre_f_b_frigo_connecte,hotel_corner_actuel_offre_f_b_reception,hotel_corner_actuel_offre_f_b_snacking_comptoir,hotel_corner_actuel_offre_non_f_b_armoire_connectee,hotel_corner_actuel_offre_non_f_b_caisse_code_barres,hotel_corner_actuel_offre_non_f_b_distributeur_auto,hotel_corner_actuel_offre_non_f_b_reception,hotel_metres_lineaires_dedies_corner
0,H2075,Ibis budget Nice Californie,IBIS BUDGET,Nice,58-60 AVENUE DE LA CALIFORNIE,NaN,06200,43.689186,7.240512,NaN,FRANCHISE,NaN,NaN,129.0,NaN,NaN,NaN,1.0,NaN,1.0,1.0,1.0,1.0,0.0,NaN,0.0,NaN,0.0,0.0,4.0,0.0,NaN,NaN,1.0,NaN,NaN,NaN,NaN,1.0,3.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,6.0
1,HB6A3,Ibis budget Strasbourg Centre République,IBIS BUDGET,Strasbourg,23A RUE OBERLIN,NaN,67000,48.591522,7.754599,2020.0,FRANCHISE,2025.0,2025.0,97.0,0.70,0.60,0.80,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.4,0.6,1.0,NaN,NaN,0.4,0.6,1.0,2.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,2.0
2,H0815,Ibis Styles Roissy CDG,IBIS STYLES,Roissy,2 AVENUE HEINZ GLOOR,NaN,95700,49.006733,2.519843,2015.0,FRANCHISE,2015.0,2015.0,309.0,0.95,0.91,0.98,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,20.0,80.0,NaN,NaN,1.0,30.0,70.0,1.0,NaN,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,5.0
3,H6188,Mercure Paris Boulogne,MERCURE,Boulogne-Billancourt,37 PLACE RENÉ CLAIR,NaN,92100,48.833827,2.256274,NaN,FRANCHISE,NaN,NaN,191.0,NaN,NaN,NaN,1.0,0.0,1.0,0.0,0.0,1.0,1.0,NaN,2.0,NaN,1.0,1.0,13.0,0.0,NaN,NaN,NaN,1.0,NaN,NaN,NaN,1.0,NaN,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,6.0
4,H0373,Mercure Paris Montmartre Sacré-Cœur,MERCURE,Paris,3 RUE CAULAINCOURT,NaN,75018,48.885048,2.329923,NaN,MANAGÉ,NaN,NaN,305.0,NaN,NaN,NaN,1.0,0.0,0.0,0.0,0.0,1.0,0.0,NaN,1.0,NaN,0.0,1.0,1.0,0.0,NaN,NaN,NaN,1.0,NaN,NaN,NaN,1.0,NaN,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,6.0
5,HB5I0,Novotel Megève Mont-Blanc,NOVOTEL,Megève,1306 ROUTE NATIONALE,LE DOMAINE DE MEZTIVA,74120,45.859165,6.619055,NaN,FRANCHISE,NaN,NaN,572.0,NaN,NaN,NaN,1.0,0.0,0.0,0.0,0.0,0.0,1.0,NaN,1.0,NaN,1.0,2.0,3.0,0.0,NaN,NaN,NaN,NaN,1.0,NaN,NaN,0.0,NaN,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,8.0
6,H3546,Novotel Paris Centre Tour Eiffel,NOVOTEL,Paris,61 QUAI DE GRENELLE,NaN,75015,48.849778,2.282836,1976.0,MANAGÉ,NaN,NaN,764.0,NaN,NaN,NaN,1.0,0.0,1.0,1.0,0.0,0.0,1.0,NaN,3.0,NaN,1.0,1.0,36.0,0.0,NaN,NaN,NaN,NaN,1.0,NaN,NaN,1.0,NaN,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,7.0


In [62]:
imputation_with_average_value = [
"hotel_corner_de_vente_actuel_metres_lineaires",
"hotel_international_pct",
"hotel_national_pct",
"hotel_affaires_pct",
"hotel_to_le_plus_bas_taux",
"hotel_to_le_plus_haut_taux",
"hotel_to_annuel",
"hotel_derniere_reno",
"hotel_lobby_derniere_reno",
"hotel_contrat_signe_annee",
]

In [63]:
import pandas as pd

# Adresses postales : on ne touche pas aux nulls
ADDRESS_COLS = {"hotel_adresse_postale_1", "hotel_adresse_postale_2"}

# Colonnes texte / identifiants : pas d'imputation
INFO_COLS = {
    "hotel_code",
    "hotel_name",
    "hotel_brand",
    "hotel_city",
    "hotel_code_postal",
    "hotel_contrat_type",
}

# Moyenne pour les indicateurs continus identifiés
for col in imputation_with_average_value:
    if col in hotel_features.columns:
        hotel_features[col] = hotel_features[col].fillna(hotel_features[col].mean())

# Tout le reste (numérique) → 0
zero_cols = [
    c
    for c in hotel_features.columns
    if c not in ADDRESS_COLS
    and c not in INFO_COLS
    and c not in imputation_with_average_value
    and pd.api.types.is_numeric_dtype(hotel_features[c])
]
hotel_features[zero_cols] = hotel_features[zero_cols].fillna(0)

remaining_nulls = hotel_features.isna().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0].sort_values(ascending=False)
print(f"Nulls restants : {int(hotel_features.isna().sum().sum())}")
if not remaining_nulls.empty:
    print(remaining_nulls)
hotel_features

Nulls restants : 6
hotel_adresse_postale_2    6
dtype: int64


,hotel_code,hotel_name,hotel_brand,hotel_city,hotel_adresse_postale_1,hotel_adresse_postale_2,hotel_code_postal,hotel_lat,hotel_lon,hotel_contrat_signe_annee,hotel_contrat_type,hotel_derniere_reno,hotel_lobby_derniere_reno,hotel_nb_chambres,hotel_to_annuel,hotel_to_le_plus_bas_taux,hotel_to_le_plus_haut_taux,hotel_dispo_dans_lobby_assises,hotel_dispo_dans_lobby_bouilloire,hotel_dispo_dans_lobby_fontaine_a_eau,hotel_dispo_dans_lobby_machine_a_cafe,hotel_dispo_dans_lobby_micro_ondes,hotel_dispo_dans_lobby_vitrine_refrigeree,hotel_f_b_bar,hotel_f_b_minibar,hotel_f_b_restaurant,hotel_f_b_room_service,hotel_non_f_b_piscine,hotel_non_f_b_salle_de_sport,hotel_non_f_b_salles_de_reunion,hotel_non_f_b_spa,hotel_affaires_pct,hotel_loisirs_pct,hotel_loisirs_top_1_amis,hotel_loisirs_top_1_couples,hotel_loisirs_top_1_familles,hotel_international_pct,hotel_national_pct,hotel_corner_actuel_existe_deja,hotel_corner_de_vente_actuel_metres_lineaires,hotel_corner_actuel_offre_f_b_caisse_code_barres,hotel_corner_actuel_offre_f_b_distributeur_auto,hotel_corner_actuel_offre_f_b_frigo_connecte,hotel_corner_actuel_offre_f_b_reception,hotel_corner_actuel_offre_f_b_snacking_comptoir,hotel_corner_actuel_offre_non_f_b_armoire_connectee,hotel_corner_actuel_offre_non_f_b_caisse_code_barres,hotel_corner_actuel_offre_non_f_b_distributeur_auto,hotel_corner_actuel_offre_non_f_b_reception,hotel_metres_lineaires_dedies_corner
0,H2075,Ibis budget Nice Californie,IBIS BUDGET,Nice,58-60 AVENUE DE LA CALIFORNIE,NaN,06200,43.689186,7.240512,2003.666667,FRANCHISE,2020.0,2020.0,129.0,0.825,0.755,0.89,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,10.2,0.0,1.0,0.0,0.0,15.2,35.3,1.0,3.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,6.0
1,HB6A3,Ibis budget Strasbourg Centre République,IBIS BUDGET,Strasbourg,23A RUE OBERLIN,NaN,67000,48.591522,7.754599,2020.000000,FRANCHISE,2025.0,2025.0,97.0,0.700,0.600,0.80,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.4,0.6,1.0,0.0,0.0,0.4,0.6,1.0,2.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,2.0
2,H0815,Ibis Styles Roissy CDG,IBIS STYLES,Roissy,2 AVENUE HEINZ GLOOR,NaN,95700,49.006733,2.519843,2015.000000,FRANCHISE,2015.0,2015.0,309.0,0.950,0.910,0.98,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,20.0,80.0,0.0,0.0,1.0,30.0,70.0,1.0,2.5,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,5.0
3,H6188,Mercure Paris Boulogne,MERCURE,Boulogne-Billancourt,37 PLACE RENÉ CLAIR,NaN,92100,48.833827,2.256274,2003.666667,FRANCHISE,2020.0,2020.0,191.0,0.825,0.755,0.89,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,2.0,0.0,1.0,1.0,13.0,0.0,10.2,0.0,0.0,1.0,0.0,15.2,35.3,1.0,2.5,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,6.0
4,H0373,Mercure Paris Montmartre Sacré-Cœur,MERCURE,Paris,3 RUE CAULAINCOURT,NaN,75018,48.885048,2.329923,2003.666667,MANAGÉ,2020.0,2020.0,305.0,0.825,0.755,0.89,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,10.2,0.0,0.0,1.0,0.0,15.2,35.3,1.0,2.5,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,6.0
5,HB5I0,Novotel Megève Mont-Blanc,NOVOTEL,Megève,1306 ROUTE NATIONALE,LE DOMAINE DE MEZTIVA,74120,45.859165,6.619055,2003.666667,FRANCHISE,2020.0,2020.0,572.0,0.825,0.755,0.89,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,2.0,3.0,0.0,10.2,0.0,0.0,0.0,1.0,15.2,35.3,0.0,2.5,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,8.0
6,H3546,Novotel Paris Centre Tour Eiffel,NOVOTEL,Paris,61 QUAI DE GRENELLE,NaN,75015,48.849778,2.282836,1976.000000,MANAGÉ,2020.0,2020.0,764.0,0.825,0.755,0.89,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,3.0,0.0,1.0,1.0,36.0,0.0,10.2,0.0,0.0,0.0,1.0,15.2,35.3,1.0,2.5,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,7.0


In [64]:
# Contrat : dummy FRANCHISE / MANAGE (ou MANAGÉ)
contrat = hotel_features["hotel_contrat_type"].astype(str).str.upper().str.strip()
hotel_features["hotel_contrat_type_franchise"] = (contrat == "FRANCHISE").astype(float)
hotel_features["hotel_contrat_type_manage"] = contrat.str.startswith("MANAG").astype(float)
hotel_features = hotel_features.drop(columns=["hotel_contrat_type"])

# Marque : dummy puis suppression de hotel_brand
brand = hotel_features["hotel_brand"].astype(str).str.upper().str.strip()
hotel_features["hotel_brand_IBIS_BUDGET"] = (brand == "IBIS BUDGET").astype(float)
hotel_features["hotel_brand_ibis_styles"] = (brand == "IBIS STYLES").astype(float)
hotel_features["hotel_brand_mercure"] = (brand == "MERCURE").astype(float)
hotel_features["hotel_brand_novotel"] = (brand == "NOVOTEL").astype(float)
hotel_features = hotel_features.drop(columns=["hotel_brand"])

# Ordre des colonnes : identifiants / adresse, puis numériques
ID_COLS = [
    "hotel_code",
    "hotel_name",
    "hotel_adresse_postale_1",
    "hotel_adresse_postale_2",
    "hotel_code_postal",
    "hotel_city",
]
numeric_cols = [
    c
    for c in hotel_features.columns
    if c not in ID_COLS and pd.api.types.is_numeric_dtype(hotel_features[c])
]
hotel_features = hotel_features[ID_COLS + numeric_cols]
hotel_features

,hotel_code,hotel_name,hotel_adresse_postale_1,hotel_adresse_postale_2,hotel_code_postal,hotel_city,hotel_lat,hotel_lon,hotel_contrat_signe_annee,hotel_derniere_reno,hotel_lobby_derniere_reno,hotel_nb_chambres,hotel_to_annuel,hotel_to_le_plus_bas_taux,hotel_to_le_plus_haut_taux,hotel_dispo_dans_lobby_assises,hotel_dispo_dans_lobby_bouilloire,hotel_dispo_dans_lobby_fontaine_a_eau,hotel_dispo_dans_lobby_machine_a_cafe,hotel_dispo_dans_lobby_micro_ondes,hotel_dispo_dans_lobby_vitrine_refrigeree,hotel_f_b_bar,hotel_f_b_minibar,hotel_f_b_restaurant,hotel_f_b_room_service,hotel_non_f_b_piscine,hotel_non_f_b_salle_de_sport,hotel_non_f_b_salles_de_reunion,hotel_non_f_b_spa,hotel_affaires_pct,hotel_loisirs_pct,hotel_loisirs_top_1_amis,hotel_loisirs_top_1_couples,hotel_loisirs_top_1_familles,hotel_international_pct,hotel_national_pct,hotel_corner_actuel_existe_deja,hotel_corner_de_vente_actuel_metres_lineaires,hotel_corner_actuel_offre_f_b_caisse_code_barres,hotel_corner_actuel_offre_f_b_distributeur_auto,hotel_corner_actuel_offre_f_b_frigo_connecte,hotel_corner_actuel_offre_f_b_reception,hotel_corner_actuel_offre_f_b_snacking_comptoir,hotel_corner_actuel_offre_non_f_b_armoire_connectee,hotel_corner_actuel_offre_non_f_b_caisse_code_barres,hotel_corner_actuel_offre_non_f_b_distributeur_auto,hotel_corner_actuel_offre_non_f_b_reception,hotel_metres_lineaires_dedies_corner,hotel_contrat_type_franchise,hotel_contrat_type_manage,hotel_brand_IBIS_BUDGET,hotel_brand_ibis_styles,hotel_brand_mercure,hotel_brand_novotel
0,H2075,Ibis budget Nice Californie,58-60 AVENUE DE LA CALIFORNIE,NaN,06200,Nice,43.689186,7.240512,2003.666667,2020.0,2020.0,129.0,0.825,0.755,0.89,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,10.2,0.0,1.0,0.0,0.0,15.2,35.3,1.0,3.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,6.0,1.0,0.0,1.0,0.0,0.0,0.0
1,HB6A3,Ibis budget Strasbourg Centre République,23A RUE OBERLIN,NaN,67000,Strasbourg,48.591522,7.754599,2020.000000,2025.0,2025.0,97.0,0.700,0.600,0.80,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.4,0.6,1.0,0.0,0.0,0.4,0.6,1.0,2.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,2.0,1.0,0.0,1.0,0.0,0.0,0.0
2,H0815,Ibis Styles Roissy CDG,2 AVENUE HEINZ GLOOR,NaN,95700,Roissy,49.006733,2.519843,2015.000000,2015.0,2015.0,309.0,0.950,0.910,0.98,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,20.0,80.0,0.0,0.0,1.0,30.0,70.0,1.0,2.5,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,5.0,1.0,0.0,0.0,1.0,0.0,0.0
3,H6188,Mercure Paris Boulogne,37 PLACE RENÉ CLAIR,NaN,92100,Boulogne-Billancourt,48.833827,2.256274,2003.666667,2020.0,2020.0,191.0,0.825,0.755,0.89,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,2.0,0.0,1.0,1.0,13.0,0.0,10.2,0.0,0.0,1.0,0.0,15.2,35.3,1.0,2.5,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,6.0,1.0,0.0,0.0,0.0,1.0,0.0
4,H0373,Mercure Paris Montmartre Sacré-Cœur,3 RUE CAULAINCOURT,NaN,75018,Paris,48.885048,2.329923,2003.666667,2020.0,2020.0,305.0,0.825,0.755,0.89,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,10.2,0.0,0.0,1.0,0.0,15.2,35.3,1.0,2.5,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,6.0,0.0,1.0,0.0,0.0,1.0,0.0
5,HB5I0,Novotel Megève Mont-Blanc,1306 ROUTE NATIONALE,LE DOMAINE DE MEZTIVA,74120,Megève,45.859165,6.619055,2003.666667,2020.0,2020.0,572.0,0.825,0.755,0.89,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,2.0,3.0,0.0,10.2,0.0,0.0,0.0,1.0,15.2,35.3,0.0,2.5,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,8.0,1.0,0.0,0.0,0.0,0.0,1.0
6,H3546,Novotel Paris Centre Tour Eiffel,61 QUAI DE GRENELLE,NaN,75015,Paris,48.849778,2.282836,1976.000000,2020.0,2020.0,764.0,0.825,0.755,0.89,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,3.0,0.0,1.0,1.0,36.0,0.0,10.2,0.0,0.0,0.0,1.0,15.2,35.3,1.0,2.5,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,7.0,0.0,1.0,0.0,0.0,0.0,1.0


In [68]:
hotel_features.to_excel("../Output/hotel_data.xlsx", index = False)

## 8. Données « marque » disponibles par source

Inventaire des fichiers Excel de base (`sources/raw`) **et** des entrées des autres étapes `prepare/`.

> **Attention** : dans `ventes.csv`, la colonne `MARQUE` désigne la **marque du produit vendu** (Coca-Cola, Mars…), pas la marque hôtel (IBIS BUDGET, NOVOTEL…).

In [73]:
import json
import openpyxl
from rod_ia.config.settings import get_settings
from rod_ia.domain.services.rod_excel_extractor import BrandProjectionsExtractor

settings = get_settings(PROJECT)
SOURCES_RAW = settings.sources_raw_dir
PREPARE_INPUTS = {
    "RodPrep": ROOT / "Input",
    "SalesPrep": PREPARE / "SalesPrep" / "Input",
    "MeteoPrep": PREPARE / "MeteoPrep" / "Input",
    "ProximityPrep": PREPARE / "ProximityPrep" / "Input",
    "AllPrep": PREPARE / "AllPrep" / "Input",
}

excel_sources = sorted(SOURCES_RAW.glob("*.xlsx")) + sorted(SOURCES_RAW.glob("*.xlsm"))
excel_sources += [p for p in PREPARE_INPUTS["RodPrep"].glob("*.xlsx") if not p.name.startswith(".~lock")]

brand_catalog = pd.DataFrame(
    [
        {
            "fichier": p.name,
            "emplacement": str(p.parent.relative_to(PROJECT)),
            "type": "Excel",
            "donnee_marque": {
                "Récapitulatif": "MARQUE par hôtel (ligne admin, colonnes hôtels)",
                "ROD - Paramètres": "Stats par marque (NB CH, RESTO, BAR, règles reco IBB/IBS/NOV/MER)",
                "ROD - Simulateurs": "Pas de marque hôtel (concepts SIMPLY/LIBERTY/CONNECTED)",
                "Analyse poids catégories": "NOM BOUTIQUE seulement (marque hôtel à déduire via registre)",
            }.get(
                next((k for k in ["Récapitulatif", "ROD - Paramètres", "ROD - Simulateurs", "Analyse"] if k.split()[0] in p.name or (k == "Récapitulatif" and "capitulatif" in p.name.lower())), ""),
                "—",
            ),
        }
        for p in excel_sources
    ]
)

other_sources = pd.DataFrame(
    [
        {
            "fichier": "001.queryVentes.csv",
            "emplacement": "sources/raw (+ SalesPrep/Input/ventes.csv)",
            "type": "CSV",
            "donnee_marque": "MARQUE = marque produit (pas hôtel) ; NOM BOUTIQUE → jointure registre",
        },
        {
            "fichier": "hotel_identity_registry.json",
            "emplacement": "data/reference",
            "type": "JSON",
            "donnee_marque": "brand par hotel_id (IBIS BUDGET, IBIS STYLES, MERCURE, NOVOTEL)",
        },
        {
            "fichier": "hotels.csv / hotels.parquet",
            "emplacement": "MeteoPrep/Input, ProximityPrep/Input",
            "type": "Parquet/CSV",
            "donnee_marque": "hotel_brand repris de RodPrep hotel_lookup",
        },
        {
            "fichier": "rod_hotel_lookup.parquet",
            "emplacement": "AllPrep/Input",
            "type": "Parquet",
            "donnee_marque": "hotel_brand + colonnes récap (marque aussi dans features)",
        },
        {
            "fichier": "brand_projections.json",
            "emplacement": "data/reference",
            "type": "JSON",
            "donnee_marque": "Extrait de ROD Paramètres : total hôtels + tranches chambres par marque",
        },
    ]
)

print("=== Fichiers Excel ===")
display(brand_catalog)
print("\n=== Autres entrées prepare / référence ===")
display(other_sources)

=== Fichiers Excel ===


,fichier,emplacement,type,donnee_marque
0,ROD - Paramètres & règles + projections nb. d'hôtels.xlsx,sources/raw,Excel,"Stats par marque (NB CH, RESTO, BAR, règles reco IBB/IBS/NOV/MER)"
1,ROD - Simulateurs + détail des coûts.xlsx,sources/raw,Excel,"Stats par marque (NB CH, RESTO, BAR, règles reco IBB/IBS/NOV/MER)"
2,Récapitulatif de l'ensemble des données ROD (2).xlsx,sources/raw,Excel,"MARQUE par hôtel (ligne admin, colonnes hôtels)"
3,Analyse du poids des catégories de produit (2024-2025).xlsm,sources/raw,Excel,—
4,recapitulatif_rod.xlsx,prepare/RodPrep/Input,Excel,"MARQUE par hôtel (ligne admin, colonnes hôtels)"



=== Autres entrées prepare / référence ===


,fichier,emplacement,type,donnee_marque
0,001.queryVentes.csv,sources/raw (+ SalesPrep/Input/ventes.csv),CSV,MARQUE = marque produit (pas hôtel) ; NOM BOUTIQUE → jointure registre
1,hotel_identity_registry.json,data/reference,JSON,"brand par hotel_id (IBIS BUDGET, IBIS STYLES, MERCURE, NOVOTEL)"
2,hotels.csv / hotels.parquet,"MeteoPrep/Input, ProximityPrep/Input",Parquet/CSV,hotel_brand repris de RodPrep hotel_lookup
3,rod_hotel_lookup.parquet,AllPrep/Input,Parquet,hotel_brand + colonnes récap (marque aussi dans features)
4,brand_projections.json,data/reference,JSON,Extrait de ROD Paramètres : total hôtels + tranches chambres par marque


In [74]:
# --- 8.1 Récapitulatif ROD : MARQUE par hôtel ---
MARQUE_COL = next(
    (c for c in hotel_lookup.columns if c.endswith("donnees_admin_marque")),
    None,
)

recap_brand = hotel_lookup[
    ["hotel_code", "hotel_name", "hotel_brand"]
    + ([MARQUE_COL] if MARQUE_COL else [])
].copy()

if MARQUE_COL:
    recap_brand["marque_recap"] = recap_brand[MARQUE_COL]
    recap_brand["aligne_registre"] = (
        recap_brand["hotel_brand"].astype(str).str.upper().str.strip()
        == recap_brand["marque_recap"].astype(str).str.upper().str.strip()
    )
    recap_brand = recap_brand.drop(columns=[MARQUE_COL])

print("MARQUE dans le récap (ligne admin, colonnes hôtels) → colonne", MARQUE_COL)
display(recap_brand)

# Lecture directe du fichier source (avant pipeline RodPrep)
recap_src = next(SOURCES_RAW.glob("*capitulatif*.xlsx"))
wb = openpyxl.load_workbook(recap_src, data_only=True, read_only=True)
ws = wb["RECAP DATA ROD"]
header_row = next(ws.iter_rows(min_row=3, max_row=3, min_col=11, values_only=True))
marque_row = None
for row in ws.iter_rows(min_row=1, max_row=500, min_col=1, max_col=25, values_only=True):
    if row[3] and str(row[3]).strip().upper() == "MARQUE":
        marque_row = row
        break
wb.close()

if marque_row:
    recap_wide = pd.DataFrame(
        {
            "colonne_recap": list(header_row),
            "marque": list(marque_row[10:10 + len(header_row)]),
        }
    ).dropna(subset=["colonne_recap"])
    print("\nVue wide Excel (sources/raw) :")
    display(recap_wide)

MARQUE dans le récap (ligne admin, colonnes hôtels) → colonne None


,hotel_code,hotel_name,hotel_brand
0,H2075,Ibis budget Nice Californie,IBIS BUDGET
1,HB6A3,Ibis budget Strasbourg Centre République,IBIS BUDGET
2,H0815,Ibis Styles Roissy CDG,IBIS STYLES
3,H6188,Mercure Paris Boulogne,MERCURE
4,H0373,Mercure Paris Montmartre Sacré-Cœur,MERCURE
5,HB5I0,Novotel Megève Mont-Blanc,NOVOTEL
6,H3546,Novotel Paris Centre Tour Eiffel,NOVOTEL
7,None,Novotel Porte d'Italie,NOVOTEL



Vue wide Excel (sources/raw) :


,colonne_recap,marque
0,NICE,IBIS BUDGET
1,STRASBOURG,IBIS BUDGET
2,PARIS CDG,IBIS STYLES
3,MEGEVE,NOVOTEL
4,TOUR EIFFEL,NOVOTEL
5,MONTMARTRE,MERCURE
6,BOULOGNE,MERCURE


In [75]:
# --- 8.2 ROD Paramètres : statistiques portefeuille par marque ---
from rod_ia.domain.rules.excel_category_coeffs import BRAND_TO_CODE

param_xlsx = next(SOURCES_RAW.glob("ROD - Param*.xlsx"))
brand_payload = BrandProjectionsExtractor(
    param_xlsx, settings.brand_projections_path
).extract()

print("Fichier :", param_xlsx.name)
print("Codes marque utilisés par les règles reco :", BRAND_TO_CODE)
display(pd.DataFrame(BRAND_TO_CODE.items(), columns=["marque_hotel", "code_regle"]))

# NB CH 1 — effectifs + tranches de chambres
nb_ch_rows = []
for marque, info in brand_payload["brands"].items():
    row = {"marque": marque, "total_hotels": info["total_hotels"]}
    row.update(info["size_bands"])
    nb_ch_rows.append(row)
print("\nFeuille NB CH 1 (→ brand_projections.json)")
display(pd.DataFrame(nb_ch_rows))

# NB CH 2 — mêmes tranches avec part relative
wb = openpyxl.load_workbook(param_xlsx, data_only=True, read_only=True)
ws = wb["NB CH 2"]
nb_ch2 = []
current = None
for label, count, share, *_ in ws.iter_rows(
    min_row=2, max_row=80, min_col=2, max_col=4, values_only=True
):
    if label in brand_payload["brands"]:
        current = str(label).strip()
    elif current and label and count is not None:
        nb_ch2.append(
            {
                "marque": current,
                "tranche": str(label).strip(),
                "nb_hotels": int(count),
                "part": float(share) if share is not None else None,
            }
        )
print("\nFeuille NB CH 2 (parts relatives)")
display(pd.DataFrame(nb_ch2))


def _brand_matrix(sheet: str, value_cols: list[str]) -> pd.DataFrame:
    wb = openpyxl.load_workbook(param_xlsx, data_only=True, read_only=True)
    ws = wb[sheet]
    rows = []
    for row in ws.iter_rows(min_row=4, max_row=9, min_col=1, max_col=6, values_only=True):
        if row[0] and str(row[0]).strip() != "Total général":
            rows.append(dict(zip(["marque", *value_cols], row[: 1 + len(value_cols)])))
    wb.close()
    return pd.DataFrame(rows)


print("\nFeuille RESTO 1 — nb restaurants par marque (0 à 3)")
display(_brand_matrix("RESTO 1", ["0_resto", "1_resto", "2_resto", "3_resto", "total"]))

print("\nFeuille BAR 1 — nb bars par marque (0 à 3)")
display(_brand_matrix("BAR 1", ["0_bar", "1_bar", "2_bar", "3_bar", "total"]))

# Règles reco concept — effectifs par code marque et par règle de taille
wb = openpyxl.load_workbook(param_xlsx, data_only=True, read_only=True)
ws = wb["REGLES POUR RECO DU CONCEPT"]
rule_blocks = []
current_rule = None
for b, d, e, f, g, h in ws.iter_rows(
    min_row=1, max_row=80, min_col=2, max_col=7, values_only=True
):
    if b and str(b).startswith("REGLE"):
        current_rule = str(b).strip()
    if f in {"IBB", "IBS", "IBIS", "NOV", "MER", "TOTAL"} and g is not None:
        try:
            rule_blocks.append(
                {
                    "regle": current_rule,
                    "critere": str(d or e or "").strip() or None,
                    "code_marque": str(f),
                    "nb_hotels": int(g),
                    "part": float(h) if h is not None else None,
                }
            )
        except (TypeError, ValueError):
            pass
wb.close()
print("\nFeuille REGLES POUR RECO DU CONCEPT — effectifs par code marque")
display(pd.DataFrame(rule_blocks))

Fichier : ROD - Paramètres & règles + projections nb. d'hôtels.xlsx
Codes marque utilisés par les règles reco : {'IBIS BUDGET': 'IBB', 'IBIS STYLES': 'IBS', 'IBIS': 'IBIS', 'NOVOTEL': 'NOV', 'MERCURE': 'MER'}


,marque_hotel,code_regle
0,IBIS BUDGET,IBB
1,IBIS STYLES,IBS
2,IBIS,IBIS
3,NOVOTEL,NOV
4,MERCURE,MER



Feuille NB CH 1 (→ brand_projections.json)


,marque,total_hotels,Plus de 300 ch.,Entre 50 et 99 ch.,Entre 100 et 149 ch.,Entre 150 et 199 ch.,Entre 0 et 49 ch.,Entre 250 et 299 ch.,Entre 200 et 249 ch.,Total général
0,IBIS BUDGET,342,1,252,31,7,45.0,3.0,3,NaN
1,IBIS STYLES,267,3,174,26,3,60.0,NaN,1,NaN
2,MERCURE,255,7,155,45,15,28.0,NaN,5,NaN
3,NOVOTEL,117,2,37,57,15,NaN,4.0,2,NaN
4,IBIS,362,7,247,45,11,45.0,4.0,3,1343.0



Feuille NB CH 2 (parts relatives)


...

,marque,0_resto,1_resto,2_resto,3_resto,total
0,IBIS,181,176,4.0,1.0,362
1,IBIS BUDGET,314,28,NaN,NaN,342
2,IBIS STYLES,181,81,4.0,1.0,267
3,MERCURE,106,140,8.0,1.0,255
4,NOVOTEL,5,102,7.0,3.0,117



Feuille BAR 1 — nb bars par marque (0 à 3)


,marque,0_bar,1_bar,2_bar,3_bar,total
0,IBIS,15,341,6.0,NaN,362
1,IBIS BUDGET,319,23,NaN,NaN,342
2,IBIS STYLES,48,218,1.0,NaN,267
3,MERCURE,27,224,3.0,1.0,255
4,NOVOTEL,1,107,9.0,NaN,117



Feuille REGLES POUR RECO DU CONCEPT — effectifs par code marque


,regle,critere,code_marque,nb_hotels,part
0,REGLE #1,None,IBB,45,NaN
1,REGLE #1,None,IBS,60,NaN
2,REGLE #1,None,IBIS,45,NaN
3,REGLE #1,None,NOV,0,NaN
4,REGLE #1,None,MER,28,NaN
5,REGLE #1,None,TOTAL,178,0.132539
6,REGLE #2,None,NOV,117,NaN
7,REGLE #2,None,MER,227,NaN
8,REGLE #2,None,TOTAL,344,0.256143


In [76]:
# --- 8.3 Registre, entrées prepare/ et fichiers sans marque hôtel directe ---

# Registre identité (source canonique hotel_id → brand)
registry_brands = registry_df[
    ["hotel_id", "name_ventes", "name_rod", "brand", "city", "nb_chambres"]
].rename(columns={"brand": "marque_registre"})
print("data/reference/hotel_identity_registry.json")
display(registry_brands)

# Chaîne prepare : hotel_brand propagé depuis RodPrep
prepare_brand_views = []
for step, path in PREPARE_INPUTS.items():
    for f in sorted(path.glob("*")):
        if f.suffix not in {".csv", ".parquet"} or f.name.startswith(".~lock"):
            continue
        df = pd.read_parquet(f) if f.suffix == ".parquet" else pd.read_csv(f)
        brand_cols = [c for c in df.columns if "brand" in c.lower() or "marque" in c.lower()]
        if not brand_cols:
            continue
        id_cols = [c for c in ("hotel_code", "hotel_id", "hotel_name") if c in df.columns]
        view = df[id_cols + brand_cols].drop_duplicates()
        view.insert(0, "etape", step)
        view.insert(1, "fichier", f.name)
        prepare_brand_views.append(view)

if prepare_brand_views:
    print("\nEntrées prepare/ avec colonne marque hôtel")
    display(pd.concat(prepare_brand_views, ignore_index=True))
else:
    print("\nAucune entrée prepare/ avec colonne marque trouvée.")

# ventes.csv — MARQUE produit vs hôtel via NOM BOUTIQUE
ventes_path = PREPARE_INPUTS["SalesPrep"] / "ventes.csv"
if ventes_path.exists():
    ventes = pd.read_csv(ventes_path, usecols=["NOM BOUTIQUE", "MARQUE"])
    ventes_sample = (
        ventes.groupby("NOM BOUTIQUE", dropna=False)["MARQUE"]
        .agg(nb_lignes="count", nb_marques_produit="nunique", exemples=lambda s: ", ".join(s.dropna().astype(str).unique()[:3]))
        .reset_index()
    )
    ventes_sample["marque_hotel_registre"] = ventes_sample["NOM BOUTIQUE"].map(
        registry_df.set_index("name_ventes")["brand"]
    )
    print("\nventes.csv — MARQUE = marque produit ; marque hôtel via registre (name_ventes)")
    display(ventes_sample)

# Analyse poids catégories — même logique (NOM BOUTIQUE + MARQUE produit)
analyse_xlsm = next(SOURCES_RAW.glob("Analyse*.xlsm"), None)
if analyse_xlsm:
    analyse = pd.read_excel(analyse_xlsm, sheet_name="BASE", usecols=["NOM BOUTIQUE", "MARQUE"])
    analyse_sample = (
        analyse.groupby("NOM BOUTIQUE")["MARQUE"]
        .agg(nb_lignes="count", top_marque_produit=lambda s: s.value_counts().index[0])
        .reset_index()
    )
    analyse_sample["marque_hotel_registre"] = analyse_sample["NOM BOUTIQUE"].map(
        registry_df.set_index("name_ventes")["brand"]
    )
    print(f"\n{analyse_xlsm.name} — pas de marque hôtel ; jointure via NOM BOUTIQUE")
    display(analyse_sample)

# Simulateurs — concepts retail, pas de dimension marque hôtel
sim_xlsx = next(SOURCES_RAW.glob("ROD - Simulateurs*.xlsx"), None)
if sim_xlsx:
    wb = openpyxl.load_workbook(sim_xlsx, read_only=True)
    print(f"\n{sim_xlsx.name} — feuilles (concepts / coûts, sans marque hôtel) :")
    print(wb.sheetnames)
    wb.close()

print("\nSynthèse :")
print("  • Marque hôtel explicite : récap MARQUE, registre, hotel_brand (RodPrep → Meteo/AllPrep)")
print("  • Stats portefeuille par marque : ROD Paramètres (NB CH, RESTO, BAR, règles reco)")
print("  • Marque produit seulement : ventes.csv, Analyse poids catégories (MARQUE colonne)")
print("  • Pas de marque hôtel : ROD Simulateurs (SIMPLY / LIBERTY / CONNECTED)")

data/reference/hotel_identity_registry.json


,hotel_id,name_ventes,name_rod,marque_registre,city,nb_chambres
0,ibis-budget-nice,Ibis budget Nice,Nice Californie,IBIS BUDGET,Nice,129.0
1,ibis-budget-strasbourg,Ibis budget Strasbourg Centre République,Strasbourg République,IBIS BUDGET,Strasbourg,97.0
2,ibis-styles-roissy-cdg,None,Roissy CDG,IBIS STYLES,Roissy,309.0
3,novotel-megeve,Novotel Megève Mont-Blanc,Megève Mont Blanc,NOVOTEL,Megève,572.0
4,novotel-paris-tour-eiffel,Novotel Paris Tour Eiffel,Paris Centre Tour Eiffel,NOVOTEL,Paris,764.0
5,mercure-montmartre,Mercure Paris Montmartre Sacré-Cœur,Montmartre Sacré-Cœur,MERCURE,Paris,305.0
6,mercure-boulogne,None,Paris Boulogne,MERCURE,Boulogne-Billancourt,191.0
7,novotel-porte-italie,Novotel Porte d'Italie,None,NOVOTEL,Paris,NaN



Entrées prepare/ avec colonne marque hôtel


,etape,fichier,MARQUE,hotel_code,hotel_name,hotel_brand,d_recap_1_informations_generales_donnees_admin_marque,d_recap_1_informations_generales_donnees_admin_marque_r13
0,SalesPrep,ventes.csv,DECATHLON,NaN,NaN,NaN,NaN,NaN
1,SalesPrep,ventes.csv,DISNEY,NaN,NaN,NaN,NaN,NaN
2,SalesPrep,ventes.csv,AVRIL,NaN,NaN,NaN,NaN,NaN
3,SalesPrep,ventes.csv,DORGEVAL,NaN,NaN,NaN,NaN,NaN
4,SalesPrep,ventes.csv,MONOPRIX,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
128,AllPrep,rod_hotel_lookup.parquet,NaN,novotel-megeve,Novotel Megève Mont-Blanc,NOVOTEL,None,NOVOTEL
129,AllPrep,rod_hotel_lookup.parquet,NaN,novotel-paris-tour-eiffel,Novotel Paris Centre Tour Eiffel,NOVOTEL,None,NOVOTEL
130,AllPrep,rod_hotel_lookup.parquet,NaN,mercure-montmartre,Mercure Paris Montmartre Sacré-Cœur,MERCURE,None,MERCURE
131,AllPrep,rod_hotel_lookup.parquet,NaN,mercure-boulogne,Mercure Paris Boulogne,MERCURE,None,MERCURE


InvalidIndexError: Reindexing only valid with uniquely valued Index objects